# K-ABENA pour le Low-Resource Machine Learning

Ce notebook reproduit les expériences du Chapitre 17 dans un cadre universel :
datasets rares, CPU uniquement, langues peu dotées, données déséquilibrées.

> Ces contraintes existent sur tous les continents. K-ABENA est **plus bénéfique** là où les ressources sont limitées — le gain est inversement proportionnel aux ressources disponibles.

In [ ]:
# !pip install kabena-ml[sklearn] -q
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from kabena_ml import KabenaWrapper
from kabena_ml.core.filter import calibrate_K


## 1. Gain K-ABENA selon la taille du dataset

In [ ]:
import pandas as pd
np.random.seed(42)

results_lr = []
for n_samples in [100, 200, 500, 1000, 2000, 5000, 10000]:
    # Dataset synthétique représentatif (classification binaire)
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=n_samples, n_features=20, n_informative=15,
                                n_redundant=2, random_state=42)
    split = int(0.8 * n_samples)
    X_tr, X_te, y_tr, y_te = X[:split], X[split:], y[:split], y[split:]

    # target_pct adaptatif selon la règle low-data
    target_pct = 5 if n_samples < 200 else (7 if n_samples < 500 else 10)
    min_active  = max(5, int(0.10 * split))

    # Standard
    m_std = LogisticRegression(max_iter=500, random_state=42).fit(X_tr, y_tr)
    acc_std = accuracy_score(y_te, m_std.predict(X_te)) * 100

    # K-ABENA standard (N=0.3)
    m_ka = KabenaWrapper(LogisticRegression(max_iter=1), K='auto', N=0.3,
                         epochs=200, task='classification').fit(X_tr, y_tr)
    acc_ka = accuracy_score(y_te, m_ka.predict(X_te)) * 100

    # K-ABENA low-data (N=0.4, target_pct adaptatif)
    m_kald = KabenaWrapper(LogisticRegression(max_iter=1), K='auto', N=0.4,
                           epochs=200, task='classification').fit(X_tr, y_tr)
    acc_kald = accuracy_score(y_te, m_kald.predict(X_te)) * 100

    results_lr.append({'n': n_samples, 'Standard': acc_std,
                        'K-ABENA N=0.3': acc_ka, 'K-ABENA Low-data N=0.4': acc_kald,
                        'Δ Standard→LowData': acc_kald - acc_std,
                        'target_pct': target_pct})
    print(f'n={n_samples:5d}: std={acc_std:.1f}%  ka={acc_ka:.1f}%  '
          f'ka_ld={acc_kald:.1f}%  Δ={acc_kald-acc_std:+.1f}  (q={target_pct}%)')

df_lr = pd.DataFrame(results_lr)
print('\nGain moyen K-ABENA Low-data:', df_lr['Δ Standard→LowData'].mean().round(2), 'pts')


## 2. Entraînement offline sur CPU — mode sans connexion internet

In [ ]:
import time
from sklearn.datasets import load_breast_cancer  # dataset local, pas d'API

# Chargement de données locales (aucun appel réseau)
data = load_breast_cancer()
X_loc, y_loc = data.data, data.target
split_loc = int(0.8 * len(X_loc))
X_ltr, X_lte = X_loc[:split_loc], X_loc[split_loc:]
y_ltr, y_lte = y_loc[:split_loc], y_loc[split_loc:]

print(f'Dataset local : {len(X_loc)} exemples, {X_loc.shape[1]} features')
unique_vals, unique_cnts = np.unique(y_loc, return_counts=True)
cls_counts = dict(zip(unique_vals.tolist(), unique_cnts.tolist()))
print('Classes :', cls_counts)

# Comparaison standard vs K-ABENA (CPU uniquement)
t0 = time.time()
m_std_loc = LogisticRegression(max_iter=1000).fit(X_ltr, y_ltr)
t_std = time.time() - t0
acc_std_loc = accuracy_score(y_lte, m_std_loc.predict(X_lte)) * 100

t0 = time.time()
m_ka_loc = KabenaWrapper(
    LogisticRegression(max_iter=1), K='auto', N=0.3, epochs=500,
    task='classification', verbose=True
).fit(X_ltr, y_ltr)
t_ka = time.time() - t0
acc_ka_loc = accuracy_score(y_lte, m_ka_loc.predict(X_lte)) * 100

print(f'\nStandard  : {acc_std_loc:.2f}% | {t_std:.3f}s')
print(f'K-ABENA   : {acc_ka_loc:.2f}% | {t_ka:.3f}s')
print(f'Gain acc  : {acc_ka_loc - acc_std_loc:+.2f} pts')
print(f'Gain temps: {(1-t_ka/t_std)*100:.1f}%')
print(f'Stats K-ABENA: {m_ka_loc.stats_}')


## 3. Données déséquilibrées — K-ABENA Stratifié

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier

results_imb = []
for ratio_minority in [0.20, 0.10, 0.05, 0.02, 0.01]:
    X_im, y_im = make_classification(
        n_samples=5000, n_features=20, n_informative=15,
        weights=[1-ratio_minority, ratio_minority], random_state=42
    )
    split_im = int(0.8 * 5000)
    X_itr, X_ite = X_im[:split_im], X_im[split_im:]
    y_itr, y_ite = y_im[:split_im], y_im[split_im:]

    m_s = GradientBoostingClassifier(n_estimators=50, random_state=42).fit(X_itr, y_itr)
    f1_s = f1_score(y_ite, m_s.predict(X_ite), average='macro')

    m_k = KabenaWrapper(
        GradientBoostingClassifier(n_estimators=50, random_state=42),
        K='auto', N=0.4, stratified=True, task='classification'
    ).fit(X_itr, y_itr)
    f1_k = f1_score(y_ite, m_k.predict(X_ite), average='macro')

    results_imb.append({
        'Ratio minoritaire': f'{ratio_minority:.0%}',
        'Standard F1': f'{f1_s:.3f}',
        'K-ABENA Strat. F1': f'{f1_k:.3f}',
        'Δ F1': f'{f1_k-f1_s:+.3f}'
    })

print('Gain K-ABENA Stratifié selon le déséquilibre :')
print(pd.DataFrame(results_imb).to_string(index=False))
